# Hardware Validation — Analysis
This notebook loads the per-run mitigated hardware results and runs statistical comparisons and plots.
Generated automatically; run the cells to perform the analysis and customize figures for publication.


In [ ]:
import os, pandas as pd, json
base_dir = "reports/mitigated_hw_runs"
summary_csv = os.path.join(base_dir, "summary.csv")
metrics_csv = os.path.join(base_dir, "metrics_per_run.csv")
metrics_json = os.path.join(base_dir, "metrics_summary.json")

print("Files found:", [p for p in [summary_csv, metrics_csv, metrics_json] if os.path.exists(p)])
if os.path.exists(metrics_csv):
    metrics = pd.read_csv(metrics_csv)
    display(metrics)
else:
    print("metrics_per_run.csv not found; run src/compare_mitigated_vs_raw.py first.")

In [ ]:
# Basic statistics & plots
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
if 'metrics' in globals():
    figs = []
    for col in ['js_div','kl','l1','l2','classical_fidelity']:
        if col in metrics.columns:
            vals = metrics[col].dropna().values
            if len(vals) > 0:
                plt.figure(figsize=(5,2))
                plt.title(col)
                plt.boxplot(vals, vert=False)
                plt.savefig(os.path.join(base_dir, f"plots/{col}_boxplot.png"))
                plt.show()
    # paired test example (if you have simulator comparisons, replace arrays accordingly)
else:
    print("metrics dataframe missing.")

In [6]:
# Bootstrap CI example for classical_fidelity
import numpy as np
def bootstrap_ci(data, n_boot=10000, alpha=0.05):
    n = len(data)
    boots = []
    for _ in range(n_boot):
        s = np.random.choice(data, size=n, replace=True)
        boots.append(np.mean(s))
    lo = np.percentile(boots, 100*alpha/2)
    hi = np.percentile(boots, 100*(1-alpha/2))
    return lo, hi

if 'metrics' in globals() and 'classical_fidelity' in metrics.columns:
    arr = metrics['classical_fidelity'].dropna().values
    print("n=", len(arr), "mean=", np.mean(arr))
    lo, hi = bootstrap_ci(arr, n_boot=2000)
    print("Bootstrap 95% CI:", (lo, hi))

In [ ]:
jupyter nbconvert --to html reports/mitigated_hw_runs/02_hw_validation_analysis.ipynb --output reports/mitigated_hw_runs/02_hw_validation_analysis.html
